# Basic LLM as a Judge

Our dataset will be taken from the paper, [FineTuneBench: How well do commercial fine-tuning APIs infuse knowledge into LLMs?](https://arxiv.org/abs/2411.05059) Specifically, we will use the medical dataset. It comes in two parts: a memorization set and a generalization set.

**`medical_memorization.csv`**: 125 examples containing updated medical advice for treatments.

**`medical_generalization.csv`**: 125 examples containing vignettes to test the generalization of the fine-tuned model.

Let's look at an example from the `medical_memorization.csv` file:

In [13]:
MAX_IN_FLIGHT = 128

In [1]:
import numpy as np
import pandas as pd

# read csv
df_memo = pd.read_csv('data/medical/medical_memorization.csv')
df_gen = pd.read_csv('data/medical/medical_generalization.csv')

In [2]:
df_memo.head(3)

,Unnamed: 0,id,update_reference_snippet,updated_fact_description,prompt,answer_before,answer
0,0,0,For patients at elevated risk for perioperativ...,Previously: Functional capacity was assessed s...,What is the current recommended DASI score cut...,No specific DASI score cutoff; relied on subje...,DASI score >34 (equivalent to ≥4 METs)
1,1,1,Sodium–glucose cotransporter-2 inhibitors shou...,Previously: No specific guidance on SGLT2 inhi...,How many days before surgery should SGLT2 inhi...,No specific recommendation,3-4 days before surgery
2,2,2,"in late 2023, the U.S. FDA finally approved tw...",Previously: Renal denervation systems were not...,Which renal denervation systems are currently ...,No renal denervation systems were FDA-approved...,The Paradise (ultrasound) and Symplicity (radi...


### Memorization set

If we have a look at the memorization data, we can see that it is split into multiple columns:

`updated_reference_snippet` - The updated medical advice for the treatment.

`updated_fact_description` - This includes the previous medical advice for the treatment.

`prompt` - Essentially the question that the model is being asked.

`answer_before` - The previous medical answer before the updated knowledge.

`answer` - The actual current answer.

In [3]:
updated_fact_descriptions = df_memo['updated_fact_description'].tolist()
prompts_memo = df_memo['prompt'].tolist()
answers_memo = df_memo['answer'].tolist()

# An example
print(
    f"Prompt:\n{prompts_memo[0]}\n\n"
    f"Answer:\n{answers_memo[0]}\n\n"
    f"Fact Description:\n{updated_fact_descriptions[0]}\n\n"
)

Prompt:
What is the current recommended DASI score cutoff indicating good functional capacity for proceeding with surgery without further testing?

Answer:
DASI score >34 (equivalent to ≥4 METs)

Fact Description:
Previously: Functional capacity was assessed subjectively by the clinician. Now: The Duke Activity Status Index (DASI) is recommended as a structured tool for assessing functional capacity, with a specific cutoff of >34 (≥4 METs) indicating good functional capacity.




### Generalization set
If we look at the generalization dataset, we have the following columns:

`prompt` - A short scenario (the paper calls them vignettes) that the model is being asked to assess.

`answer_before` - The previous medical answer before the updated knowledge.

`answer` - The actual current answer.

In [4]:
prompts_gen = df_gen['prompt'].tolist()
answer_before_gen = df_gen['answer_before'].tolist()
answers_gen = df_gen['answer'].tolist()

# An example
print(
    f"Prompt:\n{prompts_gen[0]}\n\n"
    f"Answer:\n{answers_gen[0]}\n\n"
    f"Answer Before Update:\n{answer_before_gen[0]}\n\n"
    f"Fact Description:\n{updated_fact_descriptions[0]}\n\n"
)

Prompt:
A 65-year-old man is scheduled for elective cholecystectomy. He has hypertension and diabetes. His DASI score is 36, and he has no cardiac symptoms. Based on current guidelines, what is the appropriate next step in his preoperative cardiac evaluation?

Answer:
Proceed directly to surgery without further cardiac testing, as DASI score >34 indicates adequate functional capacity

Answer Before Update:
Proceed with subjective assessment of functional capacity and consider stress testing based on clinical judgment

Fact Description:
Previously: Functional capacity was assessed subjectively by the clinician. Now: The Duke Activity Status Index (DASI) is recommended as a structured tool for assessing functional capacity, with a specific cutoff of >34 (≥4 METs) indicating good functional capacity.




Ultimately, the generalization dataset aims to test whether the model knows how to handle situations it hasn't seen before, by applying the new knowledge that it has gained.

## Initial model assessment
In order to determine how successful we have been, we need to establish a baseline. To do this, we will actually use an LLM as a judge. This will help us control for things like the answers not exactly matching, but being close enough. It is more effective than simply doing string matching.

In [5]:
from template_manager import TemplateManager
from openai import OpenAI
import dotenv
from rich.pretty import pprint
from tqdm import tqdm

BASE_URL = dotenv.get_key(dotenv.find_dotenv(), "BASE_URL")
API_KEY = dotenv.get_key(dotenv.find_dotenv(), "API_KEY")

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

We build a simple `LLMJudge` class that is as general as possible. It should just be initialized with a model and a path to wherever the prompts are stored.

When we call the judge, it just takes in the prompt and answer, and returns the score. The call is just essentially a generic completion function. We also have a method that can take a list of prompts and answers and return the scores. There is nothing particularly fancy about this class in its synchronous form...but below is the _asynchronous_ version. We'll talk about this in a bit...

In [14]:
import asyncio
from tqdm.asyncio import tqdm as async_tqdm
from openai import OpenAI, AsyncOpenAI


class LLMJudge:
    def __init__(self, model, prompt_path, async_client=None, semaphore=None,
                 max_concurrent=MAX_IN_FLIGHT):
        self.sync_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
        self.async_client = async_client or AsyncOpenAI(
            base_url=BASE_URL, api_key=API_KEY, max_retries=0, timeout=600.0,
        )
        self.semaphore = semaphore or asyncio.Semaphore(max_concurrent)
        self.model = model
        self.template_manager = TemplateManager(prompt_path)
        self.system_prompt = self.template_manager.render('system.jinja')
        self.input_prompt = self.template_manager.get_template('input.jinja')

    def __repr__(self):
        return (f"LLMJudge("
                f"model={repr(self.model)}, "
                f"system_prompt={self.system_prompt})")

    def _messages(self, prompt_args):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": self.input_prompt.render(prompt_args)},
        ]

    def __call__(self, prompt_args, **kwargs):
        completion = self.sync_client.chat.completions.create(
            model=self.model, messages=self._messages(prompt_args), **kwargs
        )
        return completion.choices[0].message.content

    async def _async_call(self, prompt_args, **kwargs):
        async with self.semaphore:
            try:
                completion = await self.async_client.chat.completions.create(
                    model=self.model, messages=self._messages(prompt_args), **kwargs
                )
                return completion.choices[0].message.content
            except Exception as e:
                return e

    async def judge_all(self, prompt_args_list, desc="Processing judgments", **kwargs):
        tasks = [self._async_call(a, **kwargs) for a in prompt_args_list]
        return await async_tqdm.gather(*tasks, desc=desc)


To initialize our judge, we give it a model name and a prompt path:

In [15]:
model = 'zai-org/GLM-5.3'

In [16]:
memo_judge = LLMJudge(model, './prompts/memo', max_concurrent=MAX_IN_FLIGHT)
gen_judge = LLMJudge(model, './prompts/gen', max_concurrent=MAX_IN_FLIGHT)

These prompts look like this:

#### System
---
```text
You will be provided with a question and the correct answer. You will also be provided with a recent change in medical advice. You will then be given a second answer. Please indicate whether the second answer is correct or incorrect based on the question and the correct answer, and the new medical advice. You should only output '1' for correct or '0' for incorrect.
```
---

#### Input
---
```text
# Question
{{ prompt }}

# Answer
{{ answer }}

# Advice
{{ advice }}

# Second answer
{{ completion }}
```
---

We get another model to generate a completion and then feed this into the model, along with the prompt, answer, and advice from the memorization data:

In [17]:
generated_answer = "Proceed with subjective assessment of functional capacity and consider stress testing based on clinical judgment"

memo_judge(
    {
        'prompt': prompts_memo[0],
        'answer': answers_memo[0],
        'advice' : updated_fact_descriptions[0],
        'completion': 'Proceed with subjective assessment of functional capacity and consider stress testing based on clinical judgment',

    },
    reasoning_effort='low',
)

'0'

We also need to make a couple of functions to run our data through the API. Now, we could just loop over the data and call the API, but this is slow. Instead, we can use the asyncio library and the asynchronous version of the OpenAI API. This will allow us to make multiple requests at once. Without asyncio, each prompt would have to wait for the previous one to complete before starting. With asyncio:

- Multiple API calls can be "in flight" at the same time
- The program can efficiently handle other tasks while waiting for responses
- The overall execution time for multiple prompts is significantly reduced

This is particularly useful when you need to process many prompts, as it prevents the total time from being the sum of each individual API call's duration.

You can think of it as somewhat similar to solving embarrassingly parallel problems with multithreading libraries in python, except for I/O bound tasks. In this case, the I/O bound task is the API call.

#### Regular API call

In [16]:
import time

def slow_operation_sync(task_id, client):
    print(f"Starting sync task {task_id}")
    time.sleep(2)
    return f"Result {task_id}"

def run_tasks_sync(task_ids):
    start_time = time.time()
    client = "dummy_client"
    results = [slow_operation_sync(task_id, client) for task_id in task_ids]
    end_time = time.time()
    print(f"Sync version took {end_time - start_time:.2f} seconds")
    return results

task_ids = range(5)
results_sync = run_tasks_sync(task_ids)

Starting sync task 0
Starting sync task 1
Starting sync task 2
Starting sync task 3
Starting sync task 4
Sync version took 10.00 seconds


If each call takes 2 seconds, then it makes sense that 5 calls would take 10 seconds.

#### Asynchronous API call

In [17]:
import asyncio
from tqdm.asyncio import tqdm

async def slow_operation_async(task_id, client):
    print(f"Starting async task {task_id}")
    await asyncio.sleep(2)
    return f"Result {task_id}"

async def run_tasks_async(task_ids):
    start_time = time.time()
    client = "dummy_client"
    tasks = [slow_operation_async(task_id, client) for task_id in task_ids]
    answers = await tqdm.gather(*tasks, desc="Processing async tasks")
    end_time = time.time()
    print(f"Async version took {end_time - start_time:.2f} seconds")
    return answers


results_async = await run_tasks_async(task_ids)

Processing async tasks:   0%|          | 0/5 [00:00<?, ?it/s]

Starting async task 1
Starting async task 0
Starting async task 2
Starting async task 4
Starting async task 3


Processing async tasks: 100%|██████████| 5/5 [00:02<00:00,  2.49it/s]

Async version took 2.03 seconds


So obviously if we can make 5 calls at once, then it will only take 2 seconds to run.

We still need to make an async version of the OpenAI API call function:

In [18]:
import asyncio
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as async_tqdm

async def get_completion_async(prompt, model, client, sem):
    async with sem:
        try:
            completion = await client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "Provide a succinct, single sentence response."},
                    {"role": "user", "content": prompt}
                ],
                reasoning_effort='low',
            )
            return completion.choices[0].message.content
        except Exception as e:
            return e


async def get_answers_async(prompts, client, sem, model=model, desc="Processing"):
    tasks = [get_completion_async(p, model, client, sem) for p in prompts]
    return await async_tqdm.gather(*tasks, desc=desc)


async def main(prompts_memo, prompts_gen, model=model):
    sem = asyncio.Semaphore(MAX_IN_FLIGHT)
    async with AsyncOpenAI(
        base_url=BASE_URL,
        api_key=API_KEY,
        max_retries=0,
        timeout=600.0,
    ) as client:
        return await asyncio.gather(
            get_answers_async(prompts_memo, client, sem, model=model, desc="memo"),
            get_answers_async(prompts_gen,  client, sem, model=model, desc="gen"),
        )

In [19]:
completions_memo, completions_gen = await main(prompts_memo, prompts_gen)

gen: 100%|██████████| 125/125 [00:21<00:00,  5.83it/s]


In [20]:
print(
    f"Prompt:\n{prompts_memo[0]}\n\n"
    f"Answer:\n{answers_memo[0]}\n\n"
    f"Model completion:\n{completions_memo[0]}\n\n"
    f"Updated facts:\n{updated_fact_descriptions[0]}\n\n"
)

Prompt:
What is the current recommended DASI score cutoff indicating good functional capacity for proceeding with surgery without further testing?

Answer:
DASI score >34 (equivalent to ≥4 METs)

Model completion:
A DASI score of ≥35 (indicating an estimated metabolic equivalent capacity of at least 4 METs) is the currently recommended cutoff suggesting adequate functional capacity to proceed with surgery without further cardiopulmonary exercise testing.

Updated facts:
Previously: Functional capacity was assessed subjectively by the clinician. Now: The Duke Activity Status Index (DASI) is recommended as a structured tool for assessing functional capacity, with a specific cutoff of >34 (≥4 METs) indicating good functional capacity.




In [25]:
inputs_memo = [
    {
        'prompt': prompts_memo[i],
        'answer': answers_memo[i],
        'advice': updated_fact_descriptions[i],
        'completion': completions_memo[i],
    }
    for i in range(len(prompts_memo))
]

memo_results = await memo_judge.judge_all(inputs_memo, temperature=0.0, reasoning_effort='low')

Processing judgments: 100%|██████████| 125/125 [00:04<00:00, 25.11it/s]


In [26]:
inputs_gen = [
    {
        'prompt': prompts_gen[i],
        'answer': answers_gen[i],
        'advice' : updated_fact_descriptions[i],
        'completion': completions_gen[i],
    }
    for i in range(len(prompts_gen))
]

gen_results = await gen_judge.judge_all(inputs_gen, temperature=0.0, reasoning_effort='low')

Processing judgments: 100%|██████████| 125/125 [00:05<00:00, 21.94it/s]


Just an FYI: not running the LLMJudge asynchronously will take around 2-3 **minutes** to run. Running it asynchronously will take around 5-15 **seconds**.

In [27]:
memo_results = [int(result) for result in memo_results]
gen_results = [int(result) for result in gen_results]

In [28]:
print(f"Memorization results: {sum(memo_results)/len(memo_results) * 100 :.0f}%")
print(f"Generalization results: {sum(gen_results)/len(gen_results) * 100 :.0f}%")

Memorization results: 54%
Generalization results: 57%
